# Topic 1: Apache Airflow — Introduction

> **Phase 7 → Week 14 → Topic 1**

---

## Why Orchestration?

A Spark job that runs once is a script. A Spark job that runs daily, depends on upstream data being ready, retries on failure, alerts you on issues, and has a visible history is a **pipeline**. Orchestration turns scripts into pipelines.

```
Without orchestration:   cron job → no retries → no dependencies → no monitoring
With orchestration:      DAG → task dependencies → retries → SLAs → alerts → lineage
```

---

## Interview Cheat Sheet

**Q: What is Apache Airflow and what problem does it solve?**
> Airflow is an open-source workflow orchestration platform. You define pipelines as DAGs (Directed Acyclic Graphs) of tasks in Python code. Airflow handles scheduling, dependency management (task A must complete before task B), retries with backoff, SLA monitoring, alerting, and provides a visual UI for pipeline status. It solves the problem of coordinating multi-step, multi-system data pipelines reliably.

**Q: What is a DAG in Airflow?**
> A DAG (Directed Acyclic Graph) is a Python file that defines a pipeline. Directed means tasks have dependencies (B depends on A). Acyclic means no cycles — Airflow can always determine execution order. Each node is a Task (an operator). The graph defines what runs in what order. Airflow evaluates DAG files periodically and schedules runs based on the `schedule_interval`.

**Q: What is the difference between an Operator, a Sensor, and a Hook?**
> **Operator**: defines a single task — what to DO (BashOperator runs shell, PythonOperator runs Python, SparkSubmitOperator submits a Spark job). **Sensor**: a special operator that WAITS for a condition (S3KeySensor waits for a file to appear, ExternalTaskSensor waits for another DAG to complete). **Hook**: a reusable connection to an external system (S3Hook, PostgresHook) — operators use hooks internally.

In [ ]:
print("Apache Airflow Introduction — patterns & concepts")
print("To run Airflow locally: pip install apache-airflow[amazon]")
print("Then: airflow standalone (starts scheduler + webserver + DB)")

## Part 1: DAG Structure & Core Concepts

In [ ]:
print("""
Airflow DAG — Core Structure:
══════════════════════════════════════════════════════════════

from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.sensors.s3_key_sensor import S3KeySensor
from datetime import datetime, timedelta

# Default args apply to all tasks in the DAG
default_args = {
    'owner': 'data-engineering',
    'depends_on_past': False,        # don't wait for yesterday's run to succeed
    'email': ['oncall@company.com'],
    'email_on_failure': True,
    'email_on_retry': False,
    'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'retry_exponential_backoff': True,  # 5min, 10min, 20min
    'max_retry_delay': timedelta(hours=1),
}

with DAG(
    dag_id='daily_etl',
    default_args=default_args,
    description='Daily Bronze→Silver→Gold pipeline',
    schedule_interval='0 6 * * *',   # 6am UTC daily (cron syntax)
    start_date=datetime(2024, 1, 1),
    catchup=False,                   # don't run missed historical runs
    tags=['etl', 'production'],
) as dag:

    # Sensor: wait for upstream data file to arrive
    wait_for_file = S3KeySensor(
        task_id='wait_for_source_file',
        bucket_name='my-data-bucket',
        bucket_key='raw/orders/{{ ds }}/data.parquet',  # Jinja template: ds=2024-01-15
        aws_conn_id='aws_default',
        poke_interval=60,   # check every 60 seconds
        timeout=3600,       # fail if file doesn't arrive within 1 hour
        mode='reschedule',  # release the worker slot while waiting
    )

    # Task: run bronze ingest
    bronze_ingest = BashOperator(
        task_id='bronze_ingest',
        bash_command='''
            aws emr add-steps \\
                --cluster-id {{ var.value.emr_cluster_id }} \\
                --steps Type=Spark,Name=Bronze,ActionOnFailure=CONTINUE,\
                        Args=[--py-files,s3://bucket/libs.zip,s3://bucket/bronze.py,\
                              --date,{{ ds }}]
        ''',
    )

    def run_silver(**context):
        date = context['ds']   # e.g., '2024-01-15'
        print(f"Running silver transform for date: {date}")
        # spark-submit or boto3 EMR/Glue call here

    silver_transform = PythonOperator(
        task_id='silver_transform',
        python_callable=run_silver,
        provide_context=True,
    )

    gold_build = BashOperator(
        task_id='gold_build',
        bash_command='python /opt/airflow/scripts/gold_build.py --date {{ ds }}',
    )

    # Define dependencies: wait → bronze → silver → gold
    wait_for_file >> bronze_ingest >> silver_transform >> gold_build

Key concepts:
  ds:              execution date as string 'YYYY-MM-DD' (Jinja template)
  ds_nodash:       '20240115'
  execution_date:  the logical date for this run (not wall clock time)
  next_ds:         next execution date
  catchup=False:   only run current + future intervals, skip historical
  schedule_interval: cron, timedelta, or @daily/@hourly/@weekly shortcuts
══════════════════════════════════════════════════════════════
""")

## Part 2: Operators, Sensors & Hooks

In [ ]:
print("""
Airflow Operators — Common Ones:
══════════════════════════════════════════════════════════════

Core operators:
  BashOperator        → run a shell command
  PythonOperator      → call a Python function
  DummyOperator       → placeholder / branch junction
  EmailOperator       → send an email
  BranchPythonOperator → conditional: return task_id to skip others

AWS operators (apache-airflow-providers-amazon):
  EmrCreateJobFlowOperator     → create EMR cluster
  EmrAddStepsOperator          → add Spark steps to EMR
  EmrStepSensor                → wait for EMR step to complete
  EmrTerminateJobFlowOperator  → terminate EMR cluster
  GlueJobOperator              → run a Glue ETL job
  GlueJobSensor                → wait for Glue job to finish
  S3CopyObjectOperator         → copy files between S3 paths
  AthenaOperator               → run an Athena SQL query

Sensors (wait for condition):
  S3KeySensor           → wait for S3 file to exist
  ExternalTaskSensor    → wait for another DAG's task to succeed
  HttpSensor            → wait for HTTP endpoint to return 200
  SqlSensor             → wait for SQL query to return a row

Sensor modes:
  poke:        hold the worker slot while checking (bad for long waits)
  reschedule:  release worker slot, reschedule after poke_interval (use this)

Hooks — reusable connection objects:
  S3Hook           → upload/download/list S3 objects
  PostgresHook     → run SQL against PostgreSQL
  SnowflakeHook    → connect to Snowflake
  HttpHook         → make HTTP requests

  # Usage in PythonOperator:
  def upload_to_s3(**context):
      hook = S3Hook(aws_conn_id='aws_default')
      hook.load_file(
          filename='/tmp/output.parquet',
          key=f"output/{context['ds']}/output.parquet",
          bucket_name='my-bucket'
      )
══════════════════════════════════════════════════════════════
""")

## Part 3: XComs & Task Communication

In [ ]:
print("""
XComs — Cross-Task Communication:
══════════════════════════════════════════════════════════════

XCom (cross-communication): pass small values between tasks.
Stored in Airflow's metadata DB. Not for large data (use S3 for that).

# Push from a task:
def create_cluster(**context):
    import boto3
    emr = boto3.client('emr')
    response = emr.run_job_flow(
        Name='Daily ETL',
        Instances={'MasterInstanceType': 'm5.xlarge', ...}
    )
    cluster_id = response['JobFlowId']
    # Push to XCom — available to downstream tasks
    context['ti'].xcom_push(key='cluster_id', value=cluster_id)
    return cluster_id  # PythonOperator also auto-pushes return value

# Pull in a downstream task:
def submit_step(**context):
    cluster_id = context['ti'].xcom_pull(
        task_ids='create_cluster',
        key='cluster_id'
    )
    print(f"Using cluster: {cluster_id}")
    # boto3 EMR add_steps call here

# Use in Jinja templates:
submit_step = BashOperator(
    task_id='submit',
    bash_command="aws emr add-steps --cluster-id {{ ti.xcom_pull(task_ids='create_cluster') }} ...",
)

XCom best practices:
  ✓ Pass IDs, paths, small metadata (< 1 KB)
  ✗ NEVER push large DataFrames, file contents, arrays of millions of rows
  ✗ XCom is stored in Airflow's Postgres/MySQL — size limits apply
  ✓ For large data: write to S3, push the S3 path as XCom
══════════════════════════════════════════════════════════════
""")

## Part 4: SLAs, Retries & Alerting

In [ ]:
print("""
SLAs, Retries & Alerting:
══════════════════════════════════════════════════════════════

SLA (Service Level Agreement) — task must complete within N time of schedule:

  bronze_ingest = BashOperator(
      task_id='bronze_ingest',
      bash_command='...',
      sla=timedelta(hours=2),    # this task must finish within 2h of DAG start
  )

  # DAG-level SLA miss callback:
  def sla_miss_alert(dag, task_list, blocking_task_list, slas, blocking_tis):
      send_slack_alert(f"SLA missed: {[t.task_id for t in task_list]}")

  with DAG(..., sla_miss_callback=sla_miss_alert) as dag:
      ...

Retry configuration:
  retries=3                      → retry 3 times before marking failed
  retry_delay=timedelta(min=5)   → wait 5 min between retries
  retry_exponential_backoff=True → 5min, 10min, 20min, 40min...
  max_retry_delay=timedelta(h=1) → cap at 1 hour between retries

Alerting patterns:

  # Email on failure (built-in):
  default_args = {'email': ['team@company.com'], 'email_on_failure': True}

  # Slack alert via callback:
  def task_failure_alert(context):
      task_id = context['task_instance'].task_id
      dag_id  = context['dag'].dag_id
      exec_dt = context['execution_date']
      msg = f":red_circle: FAILED: {dag_id}.{task_id} at {exec_dt}"
      requests.post(SLACK_WEBHOOK, json={'text': msg})

  bronze = PythonOperator(
      ...,
      on_failure_callback=task_failure_alert,
      on_success_callback=None,  # optional success hook
  )

  # DAG-level success/failure:
  with DAG(..., on_failure_callback=dag_failure_alert) as dag:
      ...

Monitoring dashboard:
  Airflow UI: http://localhost:8080
  Grid view: historical run status per task
  Gantt view: task timing per run
  Logs: click any task instance → Log
══════════════════════════════════════════════════════════════
""")

## Part 5: Branching & Dynamic DAGs

In [ ]:
print("""
Branching & Dynamic DAG Patterns:
══════════════════════════════════════════════════════════════

# Branching: choose which path to execute based on condition
from airflow.operators.python import BranchPythonOperator

def choose_path(**context):
    hour = context['execution_date'].hour
    if hour == 0:       # midnight: run full load
        return 'full_load'
    else:               # other hours: run incremental
        return 'incremental_load'

branch = BranchPythonOperator(
    task_id='decide_load_type',
    python_callable=choose_path,
)
full_load        = PythonOperator(task_id='full_load', ...)
incremental_load = PythonOperator(task_id='incremental_load', ...)
merge_paths      = DummyOperator(task_id='merge_paths', trigger_rule='none_failed_min_one_success')

branch >> [full_load, incremental_load] >> merge_paths

Trigger rules (when does a task run?):
  all_success:               default — all upstream succeeded
  all_failed:                all upstream failed (cleanup task)
  all_done:                  all upstream done (success or failure)
  one_success:               at least one upstream succeeded
  none_failed:               no upstream failed (skipped OK)
  none_failed_min_one_success: join after branching

Dynamic DAG generation (create DAGs programmatically):
  REGIONS = ['us-east-1', 'eu-west-1', 'ap-southeast-1']

  for region in REGIONS:
      with DAG(
          dag_id=f'etl_{region.replace("-","_")}',
          schedule_interval='@daily',
          ...
      ) as dag:
          ingest = PythonOperator(
              task_id='ingest',
              python_callable=run_ingest,
              op_kwargs={'region': region}
          )
          globals()[f'dag_{region}'] = dag  # register in Airflow

TaskGroups (Airflow 2.x) — group related tasks visually:
  from airflow.utils.task_group import TaskGroup

  with TaskGroup('bronze_layer') as bronze_group:
      ingest_orders   = BashOperator(task_id='ingest_orders', ...)
      ingest_payments = BashOperator(task_id='ingest_payments', ...)
      ingest_users    = BashOperator(task_id='ingest_users', ...)

  with TaskGroup('silver_layer') as silver_group:
      transform = PythonOperator(...)

  bronze_group >> silver_group
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Write a complete Airflow DAG that: waits for an S3 file to appear at `s3://my-bucket/raw/{{ ds }}/orders.parquet`, then runs a Python function to validate the file (count rows, check for nulls), then submits a Spark job. Add retry logic and a Slack failure alert.
2. Create a branching DAG that checks whether today is Monday (full reload from source) or any other day (incremental update). Both branches converge at a `notify_complete` task.
3. Write a dynamic DAG generator that creates one DAG per table in a list `TABLES = ['orders', 'payments', 'users']`. Each DAG runs daily and has two tasks: `ingest` → `validate`.
4. What is the difference between `sensor poke` mode and `reschedule` mode? When would poke mode cause issues on a shared Airflow cluster?
5. You have a DAG with 5 parallel tasks after a branching operator. The merge task (using `none_failed_min_one_success`) is being skipped. What is likely wrong and how do you fix it?